# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [ ]:
import os
import json
import gradio as gr
import sqlite3
from dotenv import load_dotenv
from openai import OpenAI
from scraper import fetch_website_contents

In [ ]:
load_dotenv(override=True)

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
else:
    print("API key founded")



In [ ]:
MODEL = "gpt-oss:20b"
openai = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

In [ ]:
system_prompt = """
Role: You are an expert Technical Recruiter and Job Analyst.

Task: Analyze the provided Job Description (JD) and extract all required skills (technical, soft skills, and tools). Focus on the required skill not company's benefits.

Requirements: > 1. Rank the skills in a numbered list from 1 (Most Critical) to N (Least Critical).
2. Ranking Logic: Base the rank on the frequency of mention, the "Required" vs. "Preferred" sections, and how central the skill is to the core responsibilities described.
3. Provide a brief (one-sentence) justification for why the top 3 skills were ranked highest.
"""

In [ ]:
crawl_function = {
    "name": "fetch_website_contents",
    "description": "Receive a website url and then return content of that website.",
    "parameters": {
        "type": "object",
        "properties": {
            "url": {
                "type": "string",
                "description": "An URL of the website that user want to get content",
            },
        },
        "required": ["url"],
        "additionalProperties": False
    }
}

In [ ]:
tools = [
    {"type": "function", "function": crawl_function}
]

In [ ]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]}for h in history]
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)

    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages)

    return response.choices[0].message.content
        

In [ ]:
def handle_tool_calls(message):
    responses = []

    for tool_call in message.tool_calls:
        if tool_call.function_name == "fetch_website_contents":
            arguments = json.load(tool_call.function.arguments)
            url = arguments.get("url")
            response_content = fetch_website_content(url)
            resoponses.append({
                "role": "tool",
                "content": response_content,
                
            })
    
    return responses
    

In [ ]:
gr.ChatInterface(fn=chat).launch()